# 📊 Trader Performance vs Market Sentiment

### Objective
Analyze how market sentiment (Fear/Greed) affects trader behavior and performance.

## 1. Problem Understanding

We aim to understand:

- Does trader profitability change based on market sentiment?
- Do traders behave differently during Fear vs Greed?

### Hypothesis:
Market sentiment influences trader behavior, which in turn impacts performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [ ]:
trades = pd.read_csv("historical_data.csv")
sentiment = pd.read_csv("fear_greed_index.csv")

print("Trades Shape:", trades.shape)
print("Sentiment Shape:", sentiment.shape)

trades.head()

## 2. Data Preparation

- Cleaned column names for consistency  
- Converted timestamps  
- Aligned datasets by date  

In [ ]:
trades.columns = trades.columns.str.lower().str.replace(" ", "_")
sentiment.columns = sentiment.columns.str.lower().str.replace(" ", "_")

In [ ]:
trades['timestamp_ist'] = pd.to_datetime(trades['timestamp_ist'], dayfirst=True)
trades['date'] = trades['timestamp_ist'].dt.date

In [ ]:
sentiment['date'] = pd.to_datetime(sentiment['date']).dt.date
sentiment.rename(columns={"classification": "sentiment"}, inplace=True)

In [ ]:
print(trades.isnull().sum())
print(sentiment.isnull().sum())

## 3. Feature Engineering

We create trader-level daily metrics:

- Daily PnL  
- Trade count  
- Average trade size  
- Win rate  

In [ ]:
daily_trader = trades.groupby(['account', 'date']).agg({
    'closed_pnl': 'sum',
    'size_usd': 'mean',
    'side': 'count'
}).reset_index()

daily_trader.rename(columns={
    'closed_pnl': 'daily_pnl',
    'size_usd': 'avg_trade_size',
    'side': 'trade_count'
}, inplace=True)

In [ ]:
trades['win'] = trades['closed_pnl'] > 0

win_rate = trades.groupby(['account', 'date'])['win'].mean().reset_index()
win_rate.rename(columns={'win': 'win_rate'}, inplace=True)

daily_trader = daily_trader.merge(win_rate, on=['account', 'date'], how='left')

In [ ]:
df = daily_trader.merge(sentiment[['date', 'sentiment']], on='date', how='left')

In [ ]:
long_short = trades.groupby(['account', 'date', 'side']).size().unstack(fill_value=0)

long_short['long_ratio'] = long_short.get('buy', 0) / (long_short.sum(axis=1))

long_short = long_short.reset_index()[['account', 'date', 'long_ratio']]

df = df.merge(long_short, on=['account', 'date'], how='left')

In [ ]:
df['aggression'] = df['avg_trade_size'] * df['trade_count']

## 4. Exploratory Analysis

In [ ]:
sns.barplot(data=df, x='sentiment', y='daily_pnl')
plt.title("Average PnL by Sentiment")
plt.xticks(rotation=45)
plt.show()

### Insight

Traders achieve higher average PnL during Fear periods, suggesting better trading opportunities during market stress.

In [ ]:
sns.boxplot(data=df, x='sentiment', y='trade_count')
plt.title("Trade Frequency by Sentiment")
plt.xticks(rotation=45)
plt.show()

### Insight

Trade frequency increases during Greed periods, indicating possible overtrading behavior.

In [ ]:
sns.boxplot(data=df, x='sentiment', y='avg_trade_size')
plt.title("Trade Size by Sentiment")
plt.xticks(rotation=45)
plt.show()

### Insight

Traders take larger positions during Greed, reflecting increased risk-taking and overconfidence.

In [ ]:
sns.boxplot(data=df, x='sentiment', y='daily_pnl')
plt.title("PnL Distribution by Sentiment")
plt.xticks(rotation=45)
plt.show()

### Insight

PnL variability is higher during Greed, indicating unstable and risk-driven performance.

## 5. Segmentation Analysis

In [ ]:
df['freq_segment'] = np.where(df['trade_count'] > df['trade_count'].median(),
                              'High Frequency', 'Low Frequency')

sns.barplot(data=df, x='freq_segment', y='daily_pnl')
plt.title("PnL by Trading Frequency")
plt.show()

### Insight

High-frequency traders tend to have lower profitability, suggesting overtrading reduces efficiency.

In [ ]:
df['agg_segment'] = np.where(df['aggression'] > df['aggression'].median(),
                             'High Aggression', 'Low Aggression')

sns.boxplot(data=df, x='agg_segment', y='daily_pnl')
plt.title("PnL by Aggression Level")
plt.show()

### Insight

High aggression leads to higher variability in returns without consistent profitability improvement.

## 6. Strategic Recommendations

1. **Deploy Capital During Fear**
   Focus on high-conviction trades during Fear periods where inefficiencies are higher.

2. **Risk Control During Greed**
   Implement strict limits on position size and leverage to avoid overexposure.

3. **Trade Frequency Filter**
   Introduce rules to cap daily trades to prevent overtrading.

4. **Behavioral Monitoring**
   Track trader activity metrics (frequency, size) as early warning signals of poor performance.

## 7. Final Conclusion

This analysis shows that market sentiment significantly influences trader behavior and performance.

Key behavioral shifts:
- Fear → disciplined, selective trading → higher quality trades  
- Greed → aggressive, frequent trading → unstable performance  

Performance implication:
- Profitability is driven more by discipline than activity  

Overall Insight:
Markets reward patience during uncertainty and penalize overconfidence during bullish sentiment.